# Day 8 — 模型训练: RF / GBDT / XGBoost / LightGBM

**论文对应**: 四种机器学习算法建立股价崩盘风险预测模型

**关键设计**:
- 时序划分 (防未来信息泄露)
- 类别不平衡处理 (class_weight / scale_pos_weight)
- 基准模型 + 调优模型对比

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import pickle

OUT_DIR = r"C:\Users\1\Desktop\项目\stock-data"
MODEL_DIR = r"C:\Users\1\Desktop\项目\models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Library versions:")
for lib in [np, pd]:
    print(f"  {lib.__name__}: {lib.__version__}")

In [ ]:
# ==========================================
# 第1步: 读取建模数据
# ==========================================

df = pd.read_csv(os.path.join(OUT_DIR, "day4_features.csv"))
df["Date"] = pd.to_datetime(df["Date"])

LABEL_COL = "crash_binary"  # MDD <= -15% = crash
ID_COLS = ["Date", "symbol", "close", "return", "future_mdd_20",
           "label", "crash_binary"]

# 读取特征列表 (main_pipeline.py 输出的31个技术特征)
with open(os.path.join(OUT_DIR, "v2_feature_list.txt"), "r", encoding="utf-8") as f:
    feature_cols = [line.strip() for line in f if line.strip()]

# 只保留实际存在的列
feature_cols = [c for c in feature_cols if c in df.columns]

print(f"数据: {df.shape}")
print(f"特征: {len(feature_cols)} 个")
print(f"标签分布: crash=1 占 {df[LABEL_COL].mean():.2%}")

In [ ]:
# ==========================================
# 第2步: 时序划分
# ==========================================

train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2023-12-31")

train_idx = df[df["Date"] <= train_end].index
val_idx = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].index
test_idx = df[df["Date"] > val_end].index

# 合并 train+val 用于最终训练
train_val_idx = df[df["Date"] <= val_end].index

print(f"Train (<=2021): {len(train_idx)}")
print(f"Val (2022-2023): {len(val_idx)}")
print(f"Test (2024-2025): {len(test_idx)}")

# 准备数据
X = df[feature_cols].values.astype(np.float32)
y = df[LABEL_COL].values.astype(int)

X_train = X[train_idx]
y_train = y[train_idx]
X_val = X[val_idx]
y_val = y[val_idx]
X_test = X[test_idx]
y_test = y[test_idx]

print(f"\nX_train: {X_train.shape}, y_train crash%: {y_train.mean():.2%}")
print(f"X_val:   {X_val.shape}, y_val crash%:   {y_val.mean():.2%}")
print(f"X_test:  {X_test.shape}, y_test crash%:  {y_test.mean():.2%}")

In [ ]:
# ==========================================
# 第3步: 评估函数
# ==========================================

def evaluate_model(model, X_eval, y_eval, name=""):
    """多指标评估"""
    y_prob = model.predict_proba(X_eval)[:, 1]
    y_pred = model.predict(X_eval)
    
    return {
        "model": name,
        "AUC": roc_auc_score(y_eval, y_prob),
        "F1": f1_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred),
        "Recall": recall_score(y_eval, y_pred),
        "crash_pred_pct": y_pred.mean(),
    }

def print_metrics(metrics_dict):
    """打印评估结果"""
    print(f"  {metrics_dict['model']:20s} "
          f"AUC={metrics_dict['AUC']:.4f}  "
          f"F1={metrics_dict['F1']:.4f}  "
          f"Precision={metrics_dict['Precision']:.4f}  "
          f"Recall={metrics_dict['Recall']:.4f}")

In [ ]:
# ==========================================
# 第4步: 基准模型 (默认参数)
# ==========================================

print("=" * 60)
print("基准模型 (默认参数)")
print("=" * 60)

imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()

baseline_models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=10, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "GBDT": GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        scale_pos_weight=imbalance_ratio,
        random_state=42, n_jobs=-1, eval_metric="logloss"
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        class_weight="balanced",
        random_state=42, n_jobs=-1, verbose=-1
    ),
}

results_baseline = []

for name, model in baseline_models.items():
    print(f"\n训练 {name}...")
    if name == "GBDT":
        sw = np.where(y_train == 1, imbalance_ratio, 1.0)
        model.fit(X_train, y_train, sample_weight=sw)
    else:
        model.fit(X_train, y_train)
    
    # 验证集评估
    metrics = evaluate_model(model, X_val, y_val, name)
    results_baseline.append(metrics)
    print_metrics(metrics)
    
    # 保存模型
    with open(os.path.join(MODEL_DIR, f"{name}_baseline.pkl"), "wb") as f:
        pickle.dump(model, f)

print("\n基准模型完成!")

In [ ]:
# ==========================================
# 第5步: 超参数调优 (以LightGBM为例)
# 用train做训练, val做验证
# ==========================================

print("=" * 60)
print("LightGBM 超参数调优")
print("=" * 60)

# 划分: train用于训练, val用于调参选择
X_tune = np.vstack([X_train, X_val])
y_tune = np.hstack([y_train, y_val])
test_fold = np.array([-1]*len(X_train) + [0]*len(X_val))  # -1=train, 0=val
ps = PredefinedSplit(test_fold)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [31, 63],
    "min_child_samples": [20, 50],
}

lgbm = LGBMClassifier(
    class_weight="balanced",
    random_state=42, n_jobs=-1, verbose=-1
)

grid = GridSearchCV(
    lgbm, param_grid,
    cv=ps, scoring="roc_auc",
    n_jobs=2, verbose=1
)

grid.fit(X_tune, y_tune)

print(f"\nBest params: {grid.best_params_}")
print(f"Best CV AUC: {grid.best_score_:.4f}")

In [ ]:
# ==========================================
# 第6步: 用最佳参数重新训练所有模型在(train+val)上
# ==========================================

print("=" * 60)
print("最终模型 (train+val 训练, test 评估)")
print("=" * 60)

final_models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight="balanced",
        min_samples_leaf=10, random_state=42, n_jobs=-1
    ),
    "GBDT": GradientBoostingClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        scale_pos_weight=imbalance_ratio, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, eval_metric="logloss"
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=grid.best_params_.get("n_estimators", 200),
        max_depth=grid.best_params_.get("max_depth", 6),
        learning_rate=grid.best_params_.get("learning_rate", 0.05),
        num_leaves=grid.best_params_.get("num_leaves", 31),
        min_child_samples=grid.best_params_.get("min_child_samples", 20),
        class_weight="balanced",
        random_state=42, n_jobs=-1, verbose=-1
    ),
}

# 在train+val上训练
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

results_final = []

for name, model in final_models.items():
    print(f"\n训练 {name} (最终版)...")
    if name == "GBDT":
        imb = (y_train_val == 0).sum() / (y_train_val == 1).sum()
        sw = np.where(y_train_val == 1, imb, 1.0)
        model.fit(X_train_val, y_train_val, sample_weight=sw)
    else:
        model.fit(X_train_val, y_train_val)
    
    metrics = evaluate_model(model, X_test, y_test, name)
    results_final.append(metrics)
    print_metrics(metrics)
    
    # 保存
    with open(os.path.join(MODEL_DIR, f"{name}_final.pkl"), "wb") as f:
        pickle.dump(model, f)

print("\n最终模型完成!")

In [ ]:
# ==========================================
# 第7步: 汇总对比
# ==========================================

df_results = pd.DataFrame(results_final)
print("\n========== 测试集结果对比 ==========")
print(df_results.to_string(index=False))

best_model = df_results.loc[df_results["AUC"].idxmax(), "model"]
print(f"\n最佳模型: {best_model}")

df_results.to_csv(os.path.join(OUT_DIR, "day8_model_results.csv"), index=False)
print("\nDay8 完成! 结果已保存.")